# Genomics and the Expansion of Biological Knowledge Workflow

This notebook scaffold supports the article **Genomics and the Expansion of Biological Knowledge**. It can be expanded with expression summaries, PCA-style ordination, variant summaries, population structure, sequence distance, metagenomic profiles, condition scoring, and provenance notes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
expr = pd.read_csv(article_dir / 'data' / 'expression_matrix.csv').set_index('gene')
metadata = pd.read_csv(article_dir / 'data' / 'sample_metadata.csv')
control = metadata.loc[metadata['group'] == 'control', 'sample'].tolist()
treated = metadata.loc[metadata['group'] == 'treated', 'sample'].tolist()
summary = pd.DataFrame(index=expr.index)
summary['control_mean'] = expr[control].mean(axis=1)
summary['treated_mean'] = expr[treated].mean(axis=1)
summary['log2_fc'] = np.log2((summary['treated_mean'] + 1) / (summary['control_mean'] + 1))
summary.sort_values('log2_fc', ascending=False).round(4)

In [ ]:
variant = pd.read_csv(article_dir / 'data' / 'variant_site_summary.csv')
variant['p1'] = variant['alt_count_pop1'] / variant['n_chrom_pop1']
variant['p2'] = variant['alt_count_pop2'] / variant['n_chrom_pop2']
variant['pi1'] = 2 * variant['p1'] * (1 - variant['p1'])
variant['pi2'] = 2 * variant['p2'] * (1 - variant['p2'])
variant[['locus', 'p1', 'p2', 'pi1', 'pi2', 'missing_rate']].round(4)

In [ ]:
meta = pd.read_csv(article_dir / 'data' / 'metagenomic_profile.csv')
meta['relative_abundance'] = meta['reads'] / meta['reads'].sum()
meta['functional_potential_score'] = (
    0.35 * meta['carbon_cycle_genes'] / meta['carbon_cycle_genes'].max() +
    0.35 * meta['nitrogen_cycle_genes'] / meta['nitrogen_cycle_genes'].max() +
    0.30 * meta['stress_response_genes'] / meta['stress_response_genes'].max()
)
meta.sort_values('functional_potential_score', ascending=False).round(4)

In [ ]:
condition = pd.read_csv(article_dir / 'data' / 'genomic_condition_sites.csv')
condition['genomic_condition_score'] = (
    0.16 * condition['assembly_quality'] +
    0.16 * condition['annotation_depth'] +
    0.16 * condition['variant_quality'] +
    0.14 * condition['expression_signal'] +
    0.14 * condition['population_representation'] +
    0.14 * condition['provenance_quality'] +
    0.10 * (1 - condition['bias_risk'])
)
condition.sort_values('genomic_condition_score', ascending=False).round(3)